**кластеризатор текстов**

SnowballStemmer + TFIDF + DBSCAN

_Евгений Борисов <esborisov@sevsu.ru>_

In [1]:
import numpy as np
import pandas as pd
from tqdm.cli import tqdm

pd.options.display.precision = 2 
pd.options.display.max_colwidth = 200 
tqdm.pandas()

# загрузка данных

In [2]:
# загружаем тексты
data = pd.read_pickle('../data/news.pkl.gz')
display( len(data) )
display(  data.sample(2) )

3196

,text,tag
1329,Минчанин тайно переночевал в батутном центре и стал звездой YouTube\n\n5 декабря 2016 в 9:08\n\n42.TUT.BY\n\nМинский блогер Влад Бумага вместе со своим другом провел ночь в закрытой батутной арене...,social
1755,"Boeing сократил выпуск лайнеров 777 на 40%\n\nКак сообщает Reuters, компания Boeing на 40% сократила выпуск лайнеров 777. Теперь их собирают всего 5 штук в месяц, в то время как ранее компания зая...",economics


In [3]:
display( len( data.drop_duplicates('text') ) )

3196

# токенайзер

In [8]:
# токенайзер с лемматизацией

import re
from natasha import Doc
from natasha import Segmenter
from natasha import MorphVocab
from natasha import NewsEmbedding
from natasha import NewsMorphTagger

# from nltk.corpus import stopwords as nltk_stopwords
# stopwords = set(nltk_stopwords.words('russian'))

seg = Segmenter() # базовый токенизатор
# морфологический анализ
tagger = NewsMorphTagger( NewsEmbedding() )
lvoc = MorphVocab() # лемматизатор

# def tokenizer(text,seg=seg, tagger=tagger, lvoc=lvoc, stopwords=stopwords):
def tokenizer(text,seg=seg, tagger=tagger, lvoc=lvoc):
    doc = Doc(text)
    doc.segment(seg)
    doc.tag_morph(tagger)
    for t in doc.tokens: t.lemmatize(lvoc)
        
    return [
        t.lemma for t in doc.tokens
        if not (
             False
            # or (t.lemma in stopwords) # выкидываем предлоги, союзы и т.п.  
            or re.match(r'^[^a-zA-ZЁёА-я]+$', t.lemma) # выкидываем токены не содержащие букв
            or re.match(r'^(\w)\1+$', t.lemma)  # выкидываем токены из одного повторяющегося символа
            or re.match(r'^[^a-zA-ZЁёА-я].*$', t.lemma)  # выкидываем токены начинающиеся не с буквы
        )
    ]

# выполняем частотный анализ

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
tf_model = CountVectorizer(
        min_df=.01, # выкидываем очень редкие слова
        max_df=.25, # выкидываем очень частые слова
        tokenizer=tokenizer, # ф-ция токенайзер
        token_pattern=None, # отключаем дефолтный токенайзер
        # binary=True,
    )

In [10]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# tf_model = TfidfVectorizer(
#         min_df=.01, # выкидываем очень редкие слова
#         max_df=.10, # выкидываем очень частые слова
#         use_idf=False, # не используем обратную частоту
#         norm='l2', # нормируем TF
#         tokenizer=tokenizer, # ф-ция токенайзер
#         token_pattern=None, # отключаем дефолтный токенайзер
#         ngram_range = (2,2)
#     )

In [11]:
%%time

data_tf = tf_model.fit_transform( data['text'].str.lower() )

display(data_tf.shape)

(3196, 2335)

CPU times: user 10min 43s, sys: 1.98 s, total: 10min 45s
Wall time: 3min 39s


In [12]:
vocab = sorted(tf_model.vocabulary_)
display(len(vocab))
display(vocab)

2335

['a',
 'adobe',
 'afisha',
 'apple',
 'audi',
 'auto',
 'b',
 'by',
 'c',
 'com',
 'facebook',
 'finance',
 'flash',
 'google',
 'html',
 'http',
 'i',
 'javascript',
 'journal',
 'kia',
 'lenta',
 'news',
 'object',
 'of',
 'player',
 'realty',
 'regnum',
 'reuters',
 'ru',
 's',
 'sport',
 'street',
 't',
 'telegram',
 'the',
 'times',
 'tut',
 'twitter',
 'volkswagen',
 'wall',
 'youtube',
 'абсолютно',
 'авария',
 'август',
 'авто',
 'автобус',
 'автодорога',
 'автомат',
 'автоматический',
 'автомобиль',
 'автомобильный',
 'автор',
 'агентство',
 'агрегат',
 'адвокат',
 'административный',
 'администрация',
 'адрес',
 'академия',
 'акт',
 'актер',
 'актив',
 'активно',
 'активность',
 'активный',
 'актриса',
 'актуальный',
 'акция',
 'александр',
 'алексей',
 'алеппо',
 'альбом',
 'америка',
 'американец',
 'американский',
 'анализ',
 'аналитик',
 'аналогичный',
 'анатолий',
 'английский',
 'андрей',
 'анна',
 'анонсировать',
 'антон',
 'аппарат',
 'апрель',
 'аренда',
 'арест',
 '

## кластеризируем

In [ ]:
# оценки расстояний 
from sklearn.metrics.pairwise import euclidean_distances

d = euclidean_distances(data_tf)
pd.Series(d.flatten(),name='distance').to_frame().query('distance>0.').describe([0.001,0.01,0.1]).T

,count,mean,std,min,0.1%,1%,10%,50%,max
distance,1.02e+07,21.32,14.54,1.0,4.58,6.56,10.49,16.82,154.66


In [36]:
# eps = np.quantile(d.flatten(),q=0.01)
eps  = 6.1
# eps= 0.7

In [37]:
# import numpy as np
# np.array( data_tf.todense() ) 

In [38]:
# from sklearn.cluster import estimate_bandwidth
# # (X, *, quantile=0.3, n_samples=None, random_state=0, n_jobs=None)
# estimate_bandwidth(np.array( data_tf.todense() ))

In [39]:
%%time

from sklearn.cluster import DBSCAN

data['cluster_id'] = DBSCAN(eps=eps, min_samples=3).fit(data_tf).labels_

display( data['cluster_id'].drop_duplicates().count() )

np.int64(15)

CPU times: user 537 ms, sys: 17.6 ms, total: 554 ms
Wall time: 562 ms


In [40]:
# номер кластера, количество объектов, метки объектов
# (cluster=-1 - некластеризованные DBSCAN объекты) 
cluster_descr = pd.concat([
        data[['cluster_id','tag']].groupby(['cluster_id'])['tag'].count(),
        data[['cluster_id','tag']].groupby(['cluster_id'])['tag'].apply(lambda s: set(s)).apply(' '.join)
    ],axis=1).reset_index()

cluster_descr.columns = ['cluster_id','count','tags']

display( cluster_descr )

,cluster_id,count,tags
0,-1,2668,science culture politics incident woman auto tech sport social realty reclama health economics
1,0,478,science politics culture incident auto tech sport social health economics
2,1,3,politics
3,2,3,politics
4,3,3,politics
5,4,7,politics
6,5,3,politics
7,6,5,culture
8,7,3,sport
9,8,3,incident


In [42]:
display( data.query('cluster_id==9') )

,text,tag,cluster_id
2205,После анонимного сообщения о взрывном устройстве в центре Москвы проверяют поезд сообщением Москва - Киев на Киевском вокзале. Об этом сообщает РИА Новости со ссылкой на экстренные службы города.\...,incident,9
2206,Кинологи с собаками проверяют поезд сообщением Москва–Киев на Киевском вокзале в Москве после анонимного звонка о взрывном устройстве.\n\nОб этом сообщает «РИА Новости» со ссылкой на источник в эк...,incident,9
2211,Поезд Москва — Киев на Киевском вокзале в центре столицы проверяют после анонимного сообщения о взрывном устройстве. Об этом заявил источник в экстренных службах Москвы.\n\n«Кинологи проверяют ваг...,incident,9
2213,"Во вторник утром на Киевском вокзале в центре Москвы спецслужбы проверяют поезд Москва — Киев после анонимного сообщения о взрывном устройстве, сообщает РИА Новости.\n\nПо данным экстренных служб ...",incident,9
2216,"КИШИНЕВ, 13 дек — Sputnik. Силовики во вторник утром проверяют поезд сообщением Москва — Киев на Киевском вокзале в центре Москвы после анонимного сообщения о взрывном устройстве, сообщает РИА Нов...",incident,9
2217,"На вокзал уже прибыли сотрудники всех экстренных служб / progulkipomoskve\n\nОб этом заявил источник в экстренных службах Москвы, как сообщают РИА Новости.\n\nЧитайте также В Киеве на пересадочной...",incident,9


----

In [ ]:
# import re
# from nltk.tokenize import word_tokenize as nltk_tokenize_word

# def tokenizer(text):
#     return [
#             t for t in nltk_tokenize_word( # разбиваем текст на слова
#                 re.sub(r'</?[a-z]+>',' ',text), # удаляем xml tag 
#                 language='russian'
#             ) 
#         ]

In [ ]:
# import re
# from nltk.tokenize import word_tokenize as nltk_tokenize_word
# from nltk.corpus import stopwords as nltk_stopwords

# stopwords = set(nltk_stopwords.words('russian'))

# def tokenizer(text,stopwords=stopwords):
#     return [
#             t for t in nltk_tokenize_word( # разбиваем текст на слова
#                 re.sub(r'</?[a-z]+>',' ',text), # удаляем xml tag 
#                 language='russian'
#             ) 
#             if not (
#                False
#                or (len(t)<3) # выкидываем очень короткие слова
#                or re.match(r'^[^a-zA-ZЁёА-я]+$', t) # выкидываем токены не содержащие букв
#                or re.match(r'^(\w)\1+$', t)  # выкидываем токены из одного повторяющегося символа
#                or re.match(r'^[^a-zA-ZЁёА-я].*$', t)  # выкидываем токены начинающиеся не с буквы
#                or (t in stopwords) # выкидываем предлоги, союзы и т.п.    
#             )
#         ] 

In [ ]:
# NLTK package manager
# import nltk
# nltk.download()

In [ ]:
# import re
# # from razdel import sentenize
# from razdel import tokenize
# from nltk.corpus import stopwords as nltk_stopwords
# # stopwords = set(nltk_stopwords.words('russian'))

# def tokenizer(text): #,stopwords=stopwords):
#     return [
#             t.text for t in tokenize( # разбиваем текст на слова
#                 re.sub(r'</?[a-z]+>',' ',text), # удаляем xml tag 
#             ) 
#             if not (
#                False
#                or (len(t.text)<3) # выкидываем очень короткие слова
#                or re.match(r'^[^a-zA-ZЁёА-я]+$', t.text) # выкидываем токены не содержащие букв
#                or re.match(r'^(\w)\1+$', t.text)  # выкидываем токены из одного повторяющегося символа
#                or re.match(r'^[^a-zA-ZЁёА-я].*$', t.text)  # выкидываем токены начинающиеся не с буквы
#             #    or (t.text in stopwords) # выкидываем предлоги, союзы и т.п.    
#             )
#         ] 